In [ ]:
# import pandas as pd
# import numpy as np
# import os

# # 경로 설정
# file_path = r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA 데이터.csv"

# # 1. CSV 불러오기
# df = pd.read_csv(file_path)

# # 2. 날짜 변환 및 정렬
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df.sort_values('날짜').reset_index(drop=True)

# # 3. 거래량 숫자로 변환
# df['거래량'] = df['거래량'].astype(str).str.replace("M", "").astype(float) * 1_000_000

# # 4. RSI (14일)
# delta = df["종가"].diff()
# gain = delta.where(delta > 0, 0.0)
# loss = -delta.where(delta < 0, 0.0)
# avg_gain = gain.rolling(window=14).mean()
# avg_loss = loss.rolling(window=14).mean()
# rs = avg_gain / avg_loss
# df["RSI (14일)"] = 100 - (100 / (1 + rs))

# # 5. Bollinger Bands (20일)
# rolling_mean = df["종가"].rolling(window=20).mean()
# rolling_std = df["종가"].rolling(window=20).std()
# df["볼린저밴드 상단"] = rolling_mean + (2 * rolling_std)
# df["볼린저밴드 하단"] = rolling_mean - (2 * rolling_std)

# # 6. MACD + Signal
# ema_12 = df["종가"].ewm(span=12, adjust=False).mean()
# ema_26 = df["종가"].ewm(span=26, adjust=False).mean()
# df["MACD"] = ema_12 - ema_26
# df["MACD 시그널"] = df["MACD"].ewm(span=9, adjust=False).mean()

# # 7. 이동평균선 계산 (5, 10, 20, 60, 120, 200일)
# sma_periods = [5, 10, 20, 60, 120, 200]
# for p in sma_periods:
#     df[f"SMA {p}일"] = df["종가"].rolling(window=p).mean()

# # 8. 가격 및 거래량 상승률 (2주, 3개월, 6개월, 1년)
# 기간 = {
#     "2주": 10,
#     "3개월": 63,
#     "6개월": 126,
#     "1년": 252
# }
# for 이름, 일수 in 기간.items():
#     df[f"가격 상승률 ({이름})"] = df["종가"].pct_change(periods=일수) * 100
#     df[f"거래량 상승률 ({이름})"] = df["거래량"].pct_change(periods=일수) * 100

# # 9. 저장
# save_path = os.path.join(os.path.dirname(file_path), "TSLA_지표포함_이평포함.csv")
# df.to_csv(save_path, index=False, encoding='utf-8-sig')

# print(f"✅ 모든 지표 및 이동평균선 계산 완료!\n📁 저장 경로: {save_path}")


In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# 1. 원본 CSV들이 모여 있는 Tesla 폴더
src_folder = r"C:\Users\LabPC\OneDrive\주식\Back Test\Tesla"

# 2. 결과를 저장할 폴더 (없으면 자동 생성)
out_folder = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
os.makedirs(out_folder, exist_ok=True)

# 3. 모든 CSV 파일 검색
csv_files = glob.glob(os.path.join(src_folder, "*.csv"))

for file_path in csv_files:
    # 4. 영어 헤더 읽고 바로 한국어로 컬럼명 변경
    df = pd.read_csv(file_path).rename(columns={
        'Date': '날짜',
        'Price': '종가',
        'Open': '시가',
        'High': '고가',
        'Low': '저가',
        'Vol.': '거래량',
        'Change %': '변동 %'
    })

    # 5. 날짜 파싱 및 정렬
    df['날짜'] = pd.to_datetime(df['날짜'], format='%m/%d/%Y')
    df = df.sort_values('날짜').reset_index(drop=True)

    # 6. 거래량·변동 % 숫자화
    df['거래량'] = (
        df['거래량']
        .astype(str)
        .str.replace('M', '', regex=False)
        .astype(float) * 1_000_000
    )
    df['변동 %'] = (
        df['변동 %']
        .astype(str)
        .str.replace('%', '', regex=False)
        .astype(float)
    )

    # === 지표 계산 ===

    # ▶ RSI (14일)
    delta = df['종가'].diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    df['RSI (14일)'] = 100 - (100 / (1 + avg_gain/avg_loss))

    # ▶ Bollinger Bands (20일)
    m = df['종가'].rolling(window=20).mean()
    s = df['종가'].rolling(window=20).std()
    df['볼린저밴드 상단'] = m + 2 * s
    df['볼린저밴드 하단'] = m - 2 * s

    # ▶ MACD & Signal
    ema12 = df['종가'].ewm(span=12, adjust=False).mean()
    ema26 = df['종가'].ewm(span=26, adjust=False).mean()
    df['MACD']      = ema12 - ema26
    df['MACD 시그널'] = df['MACD'].ewm(span=9, adjust=False).mean()

    # ▶ 단순 이동평균선 (5,10,20,60,120,200일)
    for p in [5, 10, 20, 60, 120, 200]:
        df[f"SMA {p}일"] = df['종가'].rolling(window=p).mean()

    # ▶ 가격·거래량 % 변화 (2주,3개월,6개월,1년)
    periods = {
        '2주': 10,
        '3개월': 63,
        '6개월': 126,
        '1년': 252
    }
    for label, span in periods.items():
        df[f'가격 상승률 ({label})']  = df['종가'].pct_change(span)  * 100
        df[f'거래량 상승률 ({label})'] = df['거래량'].pct_change(span) * 100

    # 7. 저장 (원본 파일명_indicators.csv)
    base = os.path.splitext(os.path.basename(file_path))[0]
    save_path = os.path.join(out_folder, f"{base}_지표포함.csv")
    df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"✅ Processed: {file_path} → {save_path}")
